# Lecture 1 Lab — Toolchain Setup & a First Look at Sensor Data
### 01211373 Machine Learning and Programming for Industry

| | |
|---|---|
| **Estimated time** | 2–3 hours |
| **Tools** | Python, NumPy, pandas, matplotlib, scikit-learn |
| **Submit** | This completed notebook (see §9) |

Type your name - surname and student ID in the cell below. Failure to do so resuls in -1 penalty for this lab.

In [ ]:
# Name - surname and student ID

## 1. Learning Objectives

By the end of this lab, you should be able to:

- Confirm that your Python environment has the libraries this course depends on, correctly installed.
- Load and inspect a real (or realistically simulated) sensor dataset with pandas.
- Visualize raw time-series sensor signals and compute simple summary features.
- Train and evaluate a first classifier that separates normal from faulty operating conditions.

## 2. Before You Start: Environment Setup

Create an isolated environment and install the course libraries **from a terminal, before launching this notebook**:

```bash
conda create -n mlpi python=3.11 -y
conda activate mlpi
pip install numpy pandas matplotlib scikit-learn torch jupyter scipy
```

If you prefer `venv` instead of `conda`, replace the first two lines with:

```bash
python -m venv mlpi
source mlpi/bin/activate   # Windows: mlrobotics\Scripts\activate
```

Recently, `uv` is a powerful tool that is much faster than pip. For more details, visit https://docs.astral.sh/uv/
To use it with VSC is somewhat tricky, I have to say.


```bash
uv init lab01
cd lab01
uv venv
source .venv/bin/activate  # Mac
.venv\Scripts\Activate.ps1 # Windows powershell
.venv\Scripts\activate.bat # Windows command prompt

uv add numpy pandas matplotlib scikit-learn torch jupyter scipy
# add ipykernel as a dev dependency
uv add --dev ipykernel

```
Then launch this notebook from inside that environment. 

If you want to use VSC, place the .ipynb file inside the project folder. Activate the environment, then 
```bash
code .
```

**Note :** You may want to create a project for each lab, for proper management of dependencies. 

## 3. Part A — Verify Your Environment

Run the cell below. All five lines should print a version number with no errors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("torch       :", torch.__version__)

> **If something fails to import:** double-check you activated the `mlpi` environment before launching this notebook, then re-run the `pip install` command from §2 in a terminal (not in this notebook) and restart the kernel.

## 4. Part B — Load a Sensor Dataset

___

**Instruction :** In class I will demonstrate with option B. But to receive proper credit for lab submission, you must use option A. Using option B results in -2 penalty.

___

Choose **one** of the two options below. Both lead to the same next steps: you should end this part with two 1-D signals (arrays) representing a normal and a faulty run, each at least a few thousand samples long.

**Option A — Real dataset (recommended if you have internet access).**
Use the Case Western Reserve University (CWRU) Bearing Data Center dataset — a standard reference dataset for bearing-fault detection, with vibration recordings under normal and several faulty conditions. 

Download a normal-baseline file and one faulty-bearing file (any fault size) from the CWRU Bearing Data Center website

https://engineering.case.edu/bearingdatacenter/download-data-file


, place them next to this notebook, and load them with the cell below (fill in the TODO).

**Option B — Simulated fallback (use if you cannot reach the dataset).**
Run the `generate_synthetic_vibration()` cell further down instead. It creates a smooth low-noise signal for "normal" and a signal with added impulsive spikes for "faulty" — a reasonable stand-in for early bearing-fault vibration.

In [ ]:
# Students must select this option to avoid -2 penalty.

# Option A — Real dataset. Uncomment and fill in if you're using this option;
# leave commented out (and skip to the Option B cell below) otherwise.

# from scipy.io import loadmat
#
# normal_mat = loadmat("normal_baseline.mat")
# faulty_mat = loadmat("faulty_bearing.mat")
#
# TODO: CWRU .mat files store the signal under a key like 'X097_DE_time' —
# inspect normal_mat.keys() / faulty_mat.keys() and pull out the correct
# array into normal_signal / faulty_signal below.
# normal_signal = normal_mat[...].squeeze()
# faulty_signal = faulty_mat[...].squeeze()

In [ ]:
# Option B — Simulated fallback.
def generate_synthetic_vibration(n_samples=6000, fs=2000, seed=0):
    """
    Synthesize a 'normal' and a 'faulty' vibration signal.

    'Normal' is a clean sum of a couple of low-frequency sinusoids plus mild
    noise (mimicking smooth shaft rotation). 'Faulty' adds periodic impulsive
    spikes on top of the same base signal (mimicking the impact events an
    early bearing fault produces once per rotation).
    """
    rng = np.random.default_rng(seed)
    t = np.arange(n_samples) / fs

    base = 0.6 * np.sin(2 * np.pi * 12 * t) + 0.3 * np.sin(2 * np.pi * 29 * t)
    normal_signal = base + rng.normal(0, 0.05, size=n_samples)

    faulty_signal = base + rng.normal(0, 0.05, size=n_samples)
    fault_period = fs // 8  # a spike roughly 8 times per second
    for start in range(0, n_samples, fault_period):
        width = 6
        if start + width < n_samples:
            impulse = rng.uniform(1.2, 2.0) * np.hanning(width)
            faulty_signal[start:start + width] += impulse

    return normal_signal, faulty_signal


normal_signal, faulty_signal = generate_synthetic_vibration()
print("normal_signal:", normal_signal.shape, " faulty_signal:", faulty_signal.shape)

## 5. Part C — Visualize the Signals

1. Plot the normal and faulty signals on the same time axis (two subplots, shared x-axis).
2. Slice each signal into fixed-length windows (e.g., 200 samples per window).
3. For each window, compute four summary features: mean, standard deviation, RMS, and peak-to-peak amplitude.
4. Build a small pandas DataFrame with one row per window and columns `[mean, std, rms, ptp, label]`, where `label` is 0 for normal and 1 for faulty.

In [ ]:
def plot_signals(normal_signal, faulty_signal, fs=2000):
    t_normal = np.arange(len(normal_signal)) / fs
    t_faulty = np.arange(len(faulty_signal)) / fs

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
    axes[0].plot(t_normal, normal_signal, color="#3E5C76", linewidth=0.8)
    axes[0].set_title("Normal")
    axes[0].set_ylabel("Amplitude")

    axes[1].plot(t_faulty, faulty_signal, color="#FF6A39", linewidth=0.8)
    axes[1].set_title("Faulty")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_xlabel("Time (s)")

    fig.tight_layout()
    return fig

plot_signals(normal_signal, faulty_signal)
plt.show()

In [ ]:
def extract_window_features(signal, label, window_size=200):
    """
    Slice `signal` into non-overlapping windows and compute
    [mean, std, rms, peak-to-peak] per window.
    Returns a DataFrame with columns: mean, std, rms, ptp, label
    """
    n_windows = len(signal) // window_size
    rows = []
    for i in range(n_windows):
        window = signal[i * window_size:(i + 1) * window_size]
        # TODO: compute the four features below
        mean = None   # compute mean
        std = None    # compute standard deviation
        rms = None    # compute root-mean-square
        ptp = None    # compute peak-to-peak
        rows.append({"mean": mean, "std": std, "rms": rms, "ptp": ptp, "label": label})
    return pd.DataFrame(rows)


normal_features = extract_window_features(normal_signal, label=0)
faulty_features = extract_window_features(faulty_signal, label=1)
features_df = pd.concat([normal_features, faulty_features], ignore_index=True)
features_df.head()

> **Why features, not raw signals?** A raw 200-sample window is 200 numbers per example — too many, too correlated, and too noisy for a simple classifier to use directly. Four well-chosen summary statistics capture most of what distinguishes a smooth signal from a spiky one, and this is exactly the kind of feature engineering you'll rely on throughout the classical-ML half of the course.

## 6. Part D — Train Your First Classifier

Using scikit-learn's standard `fit` / `predict` pattern. This cell will only work once you've filled in the TODOs in §5 above — `features_df` currently contains `None` values on purpose.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

X = features_df[["mean", "std", "rms", "ptp"]].values
y = features_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

clf = LogisticRegression().fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Report your test accuracy above. A well-separated synthetic dataset should reach well above 90% accuracy — if you're near 50%, double-check your feature computation and label assignment in §5 before moving on.

## 7. Reflection Questions

Answer briefly (2–3 sentences each) by editing the markdown cells below.

**1. Is this lab an example of supervised, unsupervised, or reinforcement learning? How do you know?**

*Your answer:*

**2. Why did we scale the features before training the classifier? What could go wrong if we skipped this step?**

*Your answer:*

**3. If your test accuracy were much lower than your training accuracy, what would that suggest, and what would you check first?**

*Your answer:*

**4. Name one real industrial scenario (beyond bearing faults) where this same normal-vs-faulty pipeline could apply.**

*Your answer:*

## 8. Deliverables & Submission

- This notebook, completed and able to run top-to-bottom without errors (`Kernel → Restart & Run All`).
- The two plots from §5 (raw signals, and — optionally — a scatter of at least two features colored by label).
- Your printed test accuracy and confusion matrix from §6.
- Your written answers to the four reflection questions in §7.

Submit this `.ipynb` file in google classroom before the deadline.

## 9. Grading Rubric Guide

| Component | Weight |
|---|---|
| Environment verified & dataset loaded correctly | 15% |
| Signals visualized, features computed correctly | 25% |
| Classifier trained, evaluated, results reported | 30% |
| Reflection questions — clarity and correctness | 20% |
| Notebook quality (runs cleanly top-to-bottom, reasonably organized) | 10% |


Pay attention to penalties stated at certain points in the notebook.

---
**Next week:** *A Tour of Classifiers & Data Preprocessing* — logistic regression, SVM, decision trees, and k-NN.

Generated by Claude and modified by dewdotninja

July 2026